# Stage 6: Data Analysis

**Purpose**: Statistical analysis of the generated dataset to validate processing and characterize the corpus.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage5.json`  
**Output**: Analysis results (printed/displayed)

## What this notebook does

Computes statistics for the research paper including:

### Text statistics
- Word counts (mean, median, max) for `resumen` and `texto` fields
- Vocabulary size and overlap between fields
- Character length distributions

### Parameter statistics  
- Number of unique parameters per item
- Distribution of parameter value lengths
- Coverage of parameter combinations

### Dataset validation
- Verifies all items have non-empty `resumen` and `texto`
- Checks for anomalies in text length distributions

These statistics inform the dataset characteristics table in the paper.

In [1]:
# Import necessary libraries
import json
import pandas as pd
from collections import Counter

# Load the JSON data
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

# Normalize JSON data into a Pandas DataFrame
def normalize_data(json_data):
    # Flatten the main structure
    items = []
    parameters = []

    for key, item in json_data.items():
        # Main item fields
        items.append({
            'item_key': key,
            'parent_key': item.get('parent_key', None),
            'ud': item.get('ud', None),
            'concept': item.get('concept', None),
            'resumen': item.get('resumen', None),
            'texto': item.get('texto', None)
        })

        # Parameters
        if 'parameters' in item:
            for param_key, param in item['parameters'].items():
                for value in param['values']:
                    parameters.append({
                        'item_key': key,
                        'parameter_key': param_key,
                        'parameter_label': param['label'],
                        'value_label': value['label'],
                        'value': value['value']
                    })

    items_df = pd.DataFrame(items)
    parameters_df = pd.DataFrame(parameters)

    return items_df, parameters_df

# Word analysis functions
def word_stats(text):
    if not isinstance(text, str):
        return 0, set()
    words = text.split()
    return len(words), set(words)

def calculate_field_stats(data, field):
    word_counts = []
    vocab = set()
    for text in data[field].dropna():
        count, unique_words = word_stats(text)
        word_counts.append(count)
        vocab.update(unique_words)
    return {
        'total_count': len(word_counts),
        'total_words': sum(word_counts),
        'unique_words': len(vocab),
        'max_words': max(word_counts, default=0),
        'mean_words': sum(word_counts) / len(word_counts) if word_counts else 0#,
        #'vocabulary': vocab
    }

def get_vocabulary(data, field):
    """
    Extracts and returns the vocabulary of a specified field.
    """
    vocab = set()
    for text in data[field].dropna():
        _, unique_words = word_stats(text)
        vocab.update(unique_words)
    return vocab

# Analyze parameters
def analyze_parameters(parameters_df):
    stats = {}
    word_counts_value = []
    vocab_value = set()

    word_counts_label = []
    vocab_label = set()

    if not parameters_df.empty:
        for value in parameters_df['value']:
            count, unique_words = word_stats(value)
            word_counts_value.append(count)
            vocab_value.update(unique_words)

        for label in parameters_df['parameter_label']:
            count, unique_words = word_stats(label)
            word_counts_label.append(count)
            vocab_label.update(unique_words)

    stats['value_stats'] = {
        'max_words': max(word_counts_value, default=0),
        'mean_words': sum(word_counts_value) / len(word_counts_value) if word_counts_value else 0#,
        #'vocabulary': vocab_value
    }

    stats['label_stats'] = {
        'max_words': max(word_counts_label, default=0),
        'mean_words': sum(word_counts_label) / len(word_counts_label) if word_counts_label else 0#,
        #'vocabulary': vocab_label
    }

    stats['unique_labels'] = parameters_df['parameter_label'].nunique() if not parameters_df.empty else 0

    return stats

def display_analysis_results(stats):
    """
    Display the analysis results in a formatted way.
    
    Parameters:
    stats (dict): Dictionary containing the analysis statistics
    """
    print("Overall Statistics:")
    for key, value in stats.items():
        if isinstance(value, dict):
            print(f"\n{key.capitalize()}:")
            for sub_key, sub_value in value.items():
                if sub_key == 'vocabulary':
                    print(f"  {sub_key}: {len(sub_value)} unique words")
                elif isinstance(sub_value, dict):
                    print(f"  {sub_key}:")
                    for sub_stat_key, sub_stat_value in sub_value.items():
                        if sub_stat_key == 'vocabulary':
                            print(f"    {sub_stat_key}: {len(sub_stat_value)} unique words")
                        else:
                            print(f"    {sub_stat_key}: {sub_stat_value}")
                else:
                    print(f"  {sub_key}: {sub_value}")
        else:
            print(f"{key}: {value}")

# Main execution
def main(file_path):
    # Load data
    json_data = load_json(file_path)

    # Normalize data
    items_df, parameters_df = normalize_data(json_data)

    # Analyze fields
    stats = {}
    fields_to_analyze = ['concept', 'resumen', 'texto']
    for field in fields_to_analyze:
        stats[field] = calculate_field_stats(items_df, field)

    # Analyze parameters
    stats['parameters'] = analyze_parameters(parameters_df)

    # Additional statistics
    stats['parent_keys'] = items_df['parent_key'].nunique()
    stats['total_items'] = len(items_df)

    #display_analysis_results(stats)
    
    return stats, items_df, parameters_df


In [8]:
from utils import config

file_path = config.stage_path("OBRA CIVIL", 5)
main(file_path)

({'concept': {'total_count': 126938,
   'total_words': 866219,
   'unique_words': 959,
   'max_words': 24,
   'mean_words': 6.823953426082024},
  'resumen': {'total_count': 126938,
   'total_words': 2171573,
   'unique_words': 2676,
   'max_words': 39,
   'mean_words': 17.107351620476138},
  'texto': {'total_count': 126938,
   'total_words': 8026322,
   'unique_words': 4516,
   'max_words': 312,
   'mean_words': 63.23025413981629},
  'parameters': {'value_stats': {'max_words': 11,
    'mean_words': 2.5140259153609996},
   'label_stats': {'max_words': 4, 'mean_words': 2.1814158921894213},
   'unique_labels': 137},
  'parent_keys': 545,
  'total_items': 126938},
          item_key parent_key  ud  \
 0              O#         O#       
 1             OA#        OA#       
 2            OAA#       OAA#       
 3       OAA010aaa    OAA010$  m³   
 4       OAA010aab    OAA010$  m³   
 ...           ...        ...  ..   
 126933    OIA010b    OIA010$  ud   
 126934    OIA010c    OIA010$  ud  